In [1]:
# Disable all seeding for true randomness
import random, numpy as np
random.seed()
np.random.seed()


In [6]:
# Full Demo Script: BB84 (decoy-state) + QRNG + QPP + Auto-run until target key bits + Metrics
# Option C: run baseline (no PNS) and stress (PNS enabled), both using decoy-state BB84
# NOTE: This is a simulation tool for experiments and paper metrics. Do not seed RNG for real randomness.

import os
import random
import time
import math
import json
from collections import defaultdict, Counter
from typing import List
import numpy as np

# Qiskit imports (assumes qiskit and qiskit-aer are installed)
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# ---------------------------
# 1) Utility: QRNG (memory-based)
# ---------------------------
def qrng_generate_bits_qiskit(n_bits: int, backend_name='aer_simulator', use_fixed_seed: bool = False):
    """
    Returns a string of '0'/'1' of length n_bits using aer simulator memory (shot-by-shot).
    """
    backend = AerSimulator()
    max_shots_per_job = 8192
    bits = []
    remaining = n_bits
    while remaining > 0:
        shots = min(remaining, max_shots_per_job)
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        compiled = transpile(qc, backend)

        run_kwargs = {"shots": shots, "memory": True}
        if use_fixed_seed:
            run_kwargs["seed_simulator"] = 42

        job = backend.run(compiled, **run_kwargs)
        result = job.result()
        # try to get per-shot memory
        try:
            mem = result.get_memory()
        except Exception:
            counts = result.get_counts()
            mem = []
            for bitstr, cnt in counts.items():
                mem.extend([bitstr] * cnt)
            random.shuffle(mem)

        # append last character of each memory entry (single-qubit -> single char)
        for m in mem:
            bits.append(m[-1])
        remaining -= shots

    return ''.join(bits[:n_bits])


# ---------------------------
# 2) BB84 (decoy-state) simulation components (cleaned & parameterized)
# ---------------------------
def make_bb84_simulation_environment(
    N_PULSES=20_000,
    MU_SIGNAL=0.8,
    NU_DECOY=0.2,
    MU_VAC=0.0,
    intensity_probs=None,
    channel_transmittance=0.2,
    detector_efficiency=0.3,
    dark_count_prob=1e-5,
    qber_intrinsic=0.01,
    enable_pns=False,
    pns_keep_prob=1.0
):
    env = {}
    env['N_PULSES'] = N_PULSES
    env['INTENSITIES'] = {'signal': MU_SIGNAL, 'decoy': NU_DECOY, 'vacuum': MU_VAC}
    env['INTENSITY_PROBS'] = intensity_probs if intensity_probs is not None else {'signal':0.6, 'decoy':0.3, 'vacuum':0.1}
    env['CHANNEL_TRANSMITTANCE'] = channel_transmittance
    env['DETECTOR_EFFICIENCY'] = detector_efficiency
    env['DARK_COUNT_PROB'] = dark_count_prob
    env['QBER_INSTRINSIC'] = qber_intrinsic
    env['ENABLE_PNS_ATTACK'] = enable_pns
    env['PNS_KEEP_PROB'] = pns_keep_prob
    return env

# Poisson sampling
def sample_photon_number(mu: float) -> int:
    if mu <= 0:
        return 0
    return np.random.poisson(mu)

def alice_prepare(env):
    choices = list(env['INTENSITY_PROBS'].keys())
    probs = list(env['INTENSITY_PROBS'].values())
    intensity_type = np.random.choice(choices, p=probs)
    mu = env['INTENSITIES'][intensity_type]
    basis = 'Z' if random.random() < 0.5 else 'X'
    bit = 0 if random.random() < 0.5 else 1
    n_photons = sample_photon_number(mu)
    return {'intensity_type': intensity_type, 'mu': mu, 'basis': basis, 'bit': bit, 'n_photons': n_photons}

def channel_and_bob_detection(pulse, env):
    n = pulse['n_photons']
    eve_kept = 0
    if env['ENABLE_PNS_ATTACK'] and n >= 2:
        if random.random() < env['PNS_KEEP_PROB']:
            n_forward = max(0, n - 1)
            eve_kept = 1
        else:
            n_forward = n
    else:
        n_forward = n

    if n_forward > 0:
        survived = np.random.binomial(n_forward, env['CHANNEL_TRANSMITTANCE'])
    else:
        survived = 0

    if survived > 0:
        click_prob = 1 - (1 - env['DETECTOR_EFFICIENCY']) ** survived
    else:
        click_prob = 0.0
    click_prob = 1 - (1 - click_prob) * (1 - env['DARK_COUNT_PROB'])
    detected = random.random() < click_prob
    detected_bit = None
    if detected:
        error = (random.random() < env['QBER_INSTRINSIC'])
        if survived == 0 and env['DARK_COUNT_PROB'] > 0:
            detected_bit = 0 if random.random() < 0.5 else 1
        else:
            detected_bit = pulse['bit'] ^ (1 if error else 0)
    return {'detected': detected, 'detected_bit': detected_bit, 'survived_photons': survived, 'eve_kept': eve_kept}

def bob_measurement():
    return 'Z' if random.random() < 0.5 else 'X'

def run_single_pulse_bb84(env):
    pulse = alice_prepare(env)
    bob_basis = bob_measurement()
    chan = channel_and_bob_detection(pulse, env)
    basis_match = (pulse['basis'] == bob_basis)
    return {'pulse': pulse, 'bob_basis': bob_basis, 'basis_match': basis_match,
            'detected': chan['detected'], 'detected_bit': chan['detected_bit'],
            'survived_photons': chan['survived_photons'], 'eve_kept': chan['eve_kept']}

def run_bb84_simulation(env):
    events = []
    for _ in range(env['N_PULSES']):
        ev = run_single_pulse_bb84(env)
        events.append(ev)
    return events

# Extract sifted keys (Alice & Bob arrays) and select single-photon sifted indices for most secure keying
def extract_sifted_key_arrays(events):
    alice_bits = []
    bob_bits = []
    single_photon_indices = []
    for idx, ev in enumerate(events):
        if ev['basis_match'] and ev['detected']:
            alice_bits.append(ev['pulse']['bit'])
            bob_bits.append(ev['detected_bit'])
            if ev['pulse']['n_photons'] == 1:
                single_photon_indices.append(len(alice_bits)-1)  # index in sifted arrays
    return np.array(alice_bits, dtype=np.uint8), np.array(bob_bits, dtype=np.uint8), single_photon_indices

# ---------------------------
# 3) QPP functions (your code unchanged except slight defensive checks)
# ---------------------------
import hashlib

def bits_to_hexkey(bit_array: np.ndarray) -> bytes:
    n = len(bit_array)
    padded = np.pad(bit_array, (0, (8 - n%8) % 8), 'constant')
    bytes_list = []
    for i in range(0, len(padded), 8):
        byte = 0
        for j in range(8):
            byte = (byte << 1) | int(padded[i+j])
        bytes_list.append(byte)
    return bytes(bytes_list)

def derive_seed_from_key(bit_array: np.ndarray) -> int:
    key_bytes = bits_to_hexkey(bit_array)
    digest = hashlib.sha256(key_bytes).digest()
    seed_int = int.from_bytes(digest, 'big')
    return seed_int

def permutation_for_length(length: int, seed_int: int) -> List[int]:
    rng = random.Random(seed_int)
    perm = list(range(length))
    rng.shuffle(perm)
    return perm

def invert_permutation(perm: List[int]) -> List[int]:
    inv = [0]*len(perm)
    for i,p in enumerate(perm):
        inv[p] = i
    return inv

def qpp_encrypt_bytes(plaintext_bytes: bytes, key_bits: np.ndarray) -> bytes:
    seed_int = derive_seed_from_key(key_bits)
    block_size = len(key_bits) // 8
    if block_size == 0:
        raise ValueError("Key too short to form byte-level permutation. Need at least 8 bits.")
    cipher_blocks = []
    for i in range(0, len(plaintext_bytes), block_size):
        block = plaintext_bytes[i:i+block_size]
        perm = permutation_for_length(len(block), seed_int + (i//block_size))
        permuted = bytearray(len(block))
        for idx, p in enumerate(perm):
            permuted[p] = block[idx]
        cipher_blocks.append(bytes(permuted))
    return b''.join(cipher_blocks)

def qpp_decrypt_bytes(cipher_bytes: bytes, key_bits: np.ndarray) -> bytes:
    seed_int = derive_seed_from_key(key_bits)
    block_size = len(key_bits) // 8
    if block_size == 0:
        raise ValueError("Key too short to form byte-level permutation. Need at least 8 bits.")
    plain_blocks = []
    for i in range(0, len(cipher_bytes), block_size):
        block = cipher_bytes[i:i+block_size]
        perm = permutation_for_length(len(block), seed_int + (i//block_size))
        inv = invert_permutation(perm)
        unpermuted = bytearray(len(block))
        for idx, p in enumerate(inv):
            unpermuted[idx] = block[p]
        plain_blocks.append(bytes(unpermuted))
    return b''.join(plain_blocks)

# ---------------------------
# 4) Metrics functions (as provided earlier) - slightly compacted
# ---------------------------
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    assert len(a) == len(b)
    dist = 0
    for x,y in zip(a,b):
        v = x ^ y
        dist += bin(v).count("1")
    return dist

def metric_encryption_time(plaintext_bytes, key_bits):
    start = time.time()
    ciphertext = qpp_encrypt_bytes(plaintext_bytes, key_bits)
    end = time.time()
    return (end - start), ciphertext

def metric_decryption_time(ciphertext, key_bits):
    start = time.time()
    plaintext = qpp_decrypt_bytes(ciphertext, key_bits)
    end = time.time()
    return (end - start)

def metric_avalanche_effect(plaintext_bytes, key_bits):
    original_cipher = qpp_encrypt_bytes(plaintext_bytes, key_bits)
    modified = bytearray(plaintext_bytes)
    modified[0] ^= 1
    modified_cipher = qpp_encrypt_bytes(bytes(modified), key_bits)
    hd = hamming_distance_bytes(original_cipher, modified_cipher)
    total_bits = len(original_cipher) * 8 if len(original_cipher)>0 else 1
    return hd, hd/total_bits

def metric_entropy(data: bytes):
    freq = Counter(data)
    N = len(data)
    ent = 0.0
    for count in freq.values():
        p = count / N
        ent -= p * math.log2(p)
    return ent

def flip_bit(arr: np.ndarray, index: int):
    new = arr.copy()
    new[index] ^= 1
    return new

def metric_key_sensitivity(plaintext_bytes, key_bits):
    original_cipher = qpp_encrypt_bytes(plaintext_bytes, key_bits)
    modified_key = flip_bit(key_bits, 0)
    modified_cipher = qpp_encrypt_bytes(plaintext_bytes, modified_key)
    hd = hamming_distance_bytes(original_cipher, modified_cipher)
    total_bits = len(original_cipher) * 8 if len(original_cipher)>0 else 1
    return hd, hd/total_bits

def metric_frequency_test(ciphertext):
    bitstring = "".join(f"{byte:08b}" for byte in ciphertext)
    ones = bitstring.count("1")
    zeros = bitstring.count("0")
    total = len(bitstring)
    return (ones/total) if total>0 else 0.0, (zeros/total) if total>0 else 0.0

def metric_permutation_uniformity(key_bits):
    seed = derive_seed_from_key(key_bits)
    block_size = len(key_bits) // 8
    perm = permutation_for_length(block_size, seed)
    uniformity = (len(set(perm)) == block_size)
    return uniformity, perm

def run_all_metrics(plaintext: str, key_bits: np.ndarray):
    plaintext_bytes = plaintext.encode('utf-8')
    enc_t, ciphertext = metric_encryption_time(plaintext_bytes, key_bits)
    dec_t = metric_decryption_time(ciphertext, key_bits)
    hd_plain, avalanche = metric_avalanche_effect(plaintext_bytes, key_bits)
    ent = metric_entropy(ciphertext)
    hd_key, sensitivity = metric_key_sensitivity(plaintext_bytes, key_bits)
    ones_ratio, zeros_ratio = metric_frequency_test(ciphertext)
    uni, perm = metric_permutation_uniformity(key_bits)
    results = {
        "enc_time": enc_t,
        "dec_time": dec_t,
        "avalanche_hd": hd_plain,
        "avalanche_ratio": avalanche,
        "entropy": ent,
        "key_sensitivity_ratio": sensitivity,
        "freq_ones": ones_ratio,
        "freq_zeros": zeros_ratio,
        "permutation_uniform": uni,
        "permutation_sample": perm
    }
    return results

# ---------------------------
# 5) Helper: derive final_key_bits from events (single-photon sifted XOR with QRNG)
# ---------------------------
def derive_final_key_from_events(events, use_qrng=True):
    alice_sifted, bob_sifted, single_sp_indices = extract_sifted_key_arrays(events)
    total_sifted = len(alice_sifted)
    if total_sifted == 0:
        return np.array([], dtype=np.uint8), np.array([], dtype=np.uint8), np.array([], dtype=np.uint8)

    # choose positions that are single-photon sifted (most secure). If too few, fall back to all sifted
    if len(single_sp_indices) >= 1:
        # extract actual bits
        bb84_bits = np.array([alice_sifted[i] for i in single_sp_indices], dtype=np.uint8)
    else:
        bb84_bits = alice_sifted.copy()

    K_bb84 = len(bb84_bits)
    if K_bb84 == 0:
        return np.array([], dtype=np.uint8), bb84_bits, np.array([], dtype=np.uint8)

    # generate QRNG bits of length K_bb84 (as string) and convert to numpy array of bits
    if use_qrng:
        qrng_str = qrng_generate_bits_qiskit(K_bb84, use_fixed_seed=False)
    else:
        # fallback pseudorandom
        qrng_str = ''.join('1' if random.random() < 0.5 else '0' for _ in range(K_bb84))
    qrng_bits_arr = np.array([1 if c=='1' else 0 for c in qrng_str], dtype=np.uint8)

    # final key is XOR of bb84_bits and qrng_bits_arr
    final_key_bits = np.bitwise_xor(bb84_bits, qrng_bits_arr)
    return final_key_bits, bb84_bits, qrng_bits_arr

# ---------------------------
# 6) Auto-run driver for Option C (baseline + stress)
# ---------------------------
SAMPLE_PLAINTEXT = ("AES: Over the years, while certain theoretical vulnerabilities have been "
                    "identified in AES (especially with reduced-round variants), its full version "
                    "remains secure against known cryptanalytic attack vectors. Its longevity in "
                    "the field and extensive analysis by cryptographers worldwide affirm its "
                    "robust security profile.\n\n"
                    "ChaCha20: Designed as an improvement over Salsa20, ChaCha20 has proven resilient "
                    "against various cryptographic attacks. Its 20-round structure offers a robust "
                    "security margin. Moreover, its straightforward design minimizes the surface for "
                    "potential vulnerabilities.\n\n"
                    "In summary, while AES remains an industry stalwart, ChaCha20 presents a formidable "
                    "alternative, especially in scenarios where hardware support for AES is absent or "
                    "inconsistent. Its blend of consistent performance and proven security makes it "
                    "an enticing choice in modern cryptographic applications.")

RESULTS_DIR = "/mnt/data/bb84_qrng_qpp_output"

# os.makedirs(RESULTS_DIR, exist_ok=True)

def experiment_run(env, target_key_bits=128, max_tries=4, label="baseline"):
    """
    Runs the BB84 simulation (env) and attempts to derive final_key_bits >= target_key_bits.
    If insufficient, will increase env['N_PULSES'] and retry up to max_tries times.
    Returns final_key_bits (numpy array) and metrics dict.
    """
    tries = 0
    while tries < max_tries:
        print(f"\n--- Experiment {label} attempt {tries+1}/{max_tries} with N_PULSES={env['N_PULSES']} ---")
        events = run_bb84_simulation(env)
        final_key_bits, bb84_bits, qrng_bits = derive_final_key_from_events(events, use_qrng=True)
        print("Sifted key candidates (BB84 single-photon):", len(bb84_bits))
        print("Final key bits (post XOR with QRNG):", final_key_bits.size)
        if final_key_bits.size >= target_key_bits:
            print(f"Achieved desired key length ({final_key_bits.size} >= {target_key_bits}). Proceeding to metrics.")
            # save key bytes
            key_bytes = bits_to_hexkey(final_key_bits)
            key_file = os.path.join(RESULTS_DIR, f"final_key_{label}.bin")
            with open(key_file, "wb") as f:
                f.write(key_bytes)
            print("Saved final key to:", key_file)

            # run QPP metrics on sample plaintext
            metrics = run_all_metrics(SAMPLE_PLAINTEXT, final_key_bits)
            # save metrics to JSON
            metrics_file = os.path.join(RESULTS_DIR, f"metrics_{label}.json")
            with open(metrics_file, "w") as f:
                json.dump(metrics, f, indent=2)
            print("Saved metrics to:", metrics_file)
            return final_key_bits, metrics, events
        # else increase pulses and retry
        tries += 1
        env['N_PULSES'] = int(env['N_PULSES'] * 2)  # double pulses for next try
        print("Insufficient key bits; doubling N_PULSES and retrying.")
    # exhausted tries
    print("Could not reach target key bits; returning best-effort key.")
    # save whatever we have
   # File saving disabled for restricted environments
    print("Final key (hex):", key_bytes.hex())

    metrics = run_all_metrics(SAMPLE_PLAINTEXT, final_key_bits) if final_key_bits.size>0 else {}
    metrics_file = os.path.join(RESULTS_DIR, f"metrics_{label}_best.json")
    with open(metrics_file, "w") as f:
        json.dump(metrics, f, indent=2)
    return final_key_bits, metrics, events

# ---------------------------
# 7) Run Option C: baseline (no PNS) and stress (PNS enabled)
# ---------------------------
# Baseline environment (no PNS)
env_baseline = make_bb84_simulation_environment(
    N_PULSES=20_000,
    MU_SIGNAL=0.8,
    NU_DECOY=0.2,
    MU_VAC=0.0,
    intensity_probs={'signal':0.6,'decoy':0.3,'vacuum':0.1},
    channel_transmittance=0.25,
    detector_efficiency=0.4,
    dark_count_prob=1e-5,
    qber_intrinsic=0.01,
    enable_pns=False,
    pns_keep_prob=0.0
)

# Stress environment (PNS attack enabled)
env_pns = make_bb84_simulation_environment(
    N_PULSES=20_000,
    MU_SIGNAL=0.8,
    NU_DECOY=0.2,
    MU_VAC=0.0,
    intensity_probs={'signal':0.6,'decoy':0.3,'vacuum':0.1},
    channel_transmittance=0.15,
    detector_efficiency=0.25,
    dark_count_prob=1e-5,
    qber_intrinsic=0.02,
    enable_pns=True,
    pns_keep_prob=1.0
)

# Choose target key size for metrics (128 bits recommended)
TARGET_KEY_BITS = 128

print("\n=== Running baseline experiment (no PNS) ===")
final_key_baseline, metrics_baseline, events_baseline = experiment_run(env_baseline, target_key_bits=TARGET_KEY_BITS, max_tries=4, label="baseline")

print("\n=== Running PNS stress experiment (PNS enabled) ===")
final_key_pns, metrics_pns, events_pns = experiment_run(env_pns, target_key_bits=TARGET_KEY_BITS, max_tries=4, label="pns_stress")

print("\n=== Summary ===")
print("Baseline final key bits:", final_key_baseline.size)
print("Baseline metrics:", metrics_baseline)
print("PNS final key bits:", final_key_pns.size)
print("PNS metrics:", metrics_pns)

print(f"\nResults and keys saved to folder: {RESULTS_DIR}")



=== Running baseline experiment (no PNS) ===

--- Experiment baseline attempt 1/4 with N_PULSES=20000 ---
Sifted key candidates (BB84 single-photon): 256
Final key bits (post XOR with QRNG): 256
Achieved desired key length (256 >= 128). Proceeding to metrics.


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/bb84_qrng_qpp_output/final_key_baseline.bin'